In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [ ]:
import torch
import numpy as np

In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display
from PIL import Image

In [ ]:
from open_vocab_mot import DUKEMTMC_VIDEO_REID_PATH, DUKEMTMC_VIDEO_REID_SIDECAR_PATH

In [ ]:
from open_vocab_mot.data import DukeMTMCVideoDataset, DukeSplit, collate_duke_mtmc_video_ds, DukeMTMCItemBatch

In [ ]:
duke_ds = DukeMTMCVideoDataset(DUKEMTMC_VIDEO_REID_PATH, DukeSplit.TRAIN, load_image=True, verbose=True)

In [ ]:
len(duke_ds)

In [ ]:
sample = duke_ds[9003]

In [ ]:
display(sample.frame)

In [ ]:
print(sample.frame_path)

In [ ]:
from aidan_lib.models.sam3_batched_img import SAM3BatchedImageHarness

In [ ]:
sam_harness = SAM3BatchedImageHarness()

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
loader = DataLoader(duke_ds, batch_size=2, collate_fn=collate_duke_mtmc_video_ds)

In [ ]:
batch = next(iter(loader))

In [ ]:
import torchvision.transforms.functional as TF
tensor_image_batch = []
for img in batch.frames:
    tensor_image_batch.append(
        TF.to_tensor(img).cuda()
    )

In [ ]:
display(TF.to_pil_image(tensor_image_batch[0]))
display(batch.frames[1])

In [ ]:
out = sam_harness([batch.frames[0], batch.frames[1]], "Person", move_to_cpu=False)

In [ ]:
display(Image.fromarray(out[1].masks[0].cpu().numpy()))

In [ ]:
from aidan_lib.models.dino_lib import DINOv3Harness

In [ ]:
dino_harness = DINOv3Harness(checkpoint="facebook/dinov3-vitl16-pretrain-lvd1689m", max_side_len=1024)

In [ ]:
sam_output = sam_harness(tensor_image_batch, "Person", move_to_cpu=False)

In [ ]:
major_masks = []
for frame_out in sam_output:
    major_mask_idx = torch.argmax(frame_out.masks.sum((1, 2)))
    print(major_mask_idx)
    major_masks.append(frame_out.masks[major_mask_idx])

In [ ]:
display(Image.fromarray(major_masks[0].cpu().numpy()))

In [ ]:
print(len(tensor_image_batch))
print(tensor_image_batch[0].shape)
print(len(major_masks))
print(major_masks[0].shape)

In [ ]:
dino_out = dino_harness.match_bool_segmentations_to_dino(tensor_image_batch, major_masks)

In [ ]:
# from tqdm import tqdm
# import time
# from torchvision.utils import save_image
# import awkward as ak

# loader = DataLoader(duke_ds, batch_size=8, collate_fn=collate_duke_mtmc_video_ds, shuffle=False)

# time_converting_to_tensor = 0
# time_in_sam = 0
# time_finding_best_mask = 0
# time_in_dino = 0
# time_saving = 0

# already_seen_paths = set()
# current_parent_path = None
# file_dino_embeddings = {}  # Maps from file path to numpy array of embeddings
# for batch in tqdm(loader):
#     assert isinstance(batch, DukeMTMCItemBatch)
#     tensor_image_batch = []
#     start_time = time.perf_counter()
#     for img in batch.frames:
#         tensor_image_batch.append(
#             TF.to_tensor(img).cuda()
#         )
#     time_converting_to_tensor += time.perf_counter() - start_time
    
#     start_time = time.perf_counter()
#     sam_output = sam_harness(tensor_image_batch, "Person", move_to_cpu=False)
#     # sam_output = sam_harness(batch.frames, "Person", move_to_cpu=False)
#     time_in_sam += time.perf_counter() - start_time

#     start_time = time.perf_counter()
#     major_masks = []
#     for frame_out in sam_output:
#         major_mask_idx = torch.argmax(frame_out.masks.sum((1, 2)))
#         major_masks.append(frame_out.masks[major_mask_idx])
#     time_finding_best_mask += time.perf_counter() - start_time

#     start_time = time.perf_counter()
#     dino_out = dino_harness.match_bool_segmentations_to_dino(tensor_image_batch, major_masks)
#     # dino_out = dino_harness.match_bool_segmentations_to_dino(batch.frames, major_masks)
#     time_in_dino += time.perf_counter() - start_time

#     start_time = time.perf_counter()
#     for idx in range(len(batch.frame_paths)):
#         frame_path = batch.frame_paths[idx]
#         parent_path = frame_path.parent
#         frame_stem = frame_path.stem
#         major_mask = major_masks[idx]
#         dino_embedding = dino_out[idx][0]

#         if parent_path != current_parent_path:
#             print(f"Swapping to new parent path {parent_path}")
#             if parent_path in already_seen_paths:
#                 print(f"WARNING: Going back to old parent!")
            
#             if current_parent_path is None:
#                 # Then this is the first parent we are entering now. Just skip this one
#                 print(f"Starting first actual parent")
#                 current_parent_path = parent_path
#                 already_seen_paths.add(parent_path)
#             else:
#                 # Reformat array to dataset format
#                 print("Reformatting to awkward", flush=True)
#                 file_stems = []
#                 embeddings_list = []
#                 bboxs_list = []
#                 overlaps_list = []
#                 for file_stem, embedding_data in file_dino_embeddings.items():
#                     file_stems.append(file_stem)
#                     embeddings_list.append(embedding_data["embeddings"])
#                     bboxs_list.append(embedding_data["bboxs"])
#                     overlaps_list.append(embedding_data["overlaps"])
                
#                 print("Constructing awkward dataset", flush=True)
#                 dataset = ak.Array({
#                     "file_stems": file_stems,
#                     "embeddings_list": embeddings_list,
#                     "bboxs_list": bboxs_list,
#                     "overlaps_list": overlaps_list
#                 })
#                 parent_rel_path = parent_path.relative_to(DUKEMTMC_VIDEO_REID_PATH)
#                 sidecar_parent_path = DUKEMTMC_VIDEO_REID_SIDECAR_PATH / parent_rel_path
#                 embeddings_data_path = sidecar_parent_path / "embeddings_data.parquet"
#                 print("Saving dataset parquet")
#                 ak.to_parquet(dataset, embeddings_data_path)
#                 print("Done saving dataset parquet")

#                 file_dino_embeddings = {}
#                 current_parent_path = parent_path
#                 already_seen_paths.add(parent_path)

#         file_dino_embeddings[frame_stem] = {
#             "embeddings": dino_embedding.dino_embeddings.cpu().numpy(),
#             "bboxs": dino_embedding.dino_bboxes.cpu().numpy(),
#             "overlaps": dino_embedding.dino_overlaps.cpu().numpy()
#         }

#         # Get the path in the sidecar dir
#         parent_rel_path = parent_path.relative_to(DUKEMTMC_VIDEO_REID_PATH)
#         sidecar_parent_path = DUKEMTMC_VIDEO_REID_SIDECAR_PATH / parent_rel_path
#         # Ensure our parent directory exists
#         sidecar_parent_path.mkdir(parents=True, exist_ok=True)

#         # Save the mask
#         sidecar_mask_path = sidecar_parent_path / f"{frame_stem}_major_mask.png"
#         # print(f"Saving mask to {sidecar_mask_path}")
#         float_mask = major_mask.float()
#         save_image(float_mask, sidecar_mask_path)
#     time_saving += time.perf_counter() - start_time
        

#     # print(batch.frame_paths)
#     # break

In [ ]:
# print(time_converting_to_tensor)
# print(time_in_sam)
# print(time_finding_best_mask)
# print(time_in_dino)
# print(time_saving)

In [ ]:
# print(time_converting_to_tensor)
# print(time_in_sam)
# print(time_finding_best_mask)
# print(time_in_dino)

In [ ]:
# from tqdm import tqdm
# import time
# from torchvision.utils import save_image
# import awkward as ak

# loader = DataLoader(duke_ds, batch_size=8, collate_fn=collate_duke_mtmc_video_ds, shuffle=False)

# time_converting_to_tensor = 0
# time_in_sam = 0
# time_finding_best_mask = 0
# time_saving = 0

# already_seen_paths = set()
# current_parent_path = None
# file_dino_embeddings = {}  # Maps from file path to numpy array of embeddings
# for batch in tqdm(loader):
#     assert isinstance(batch, DukeMTMCItemBatch)
#     tensor_image_batch = []
#     start_time = time.perf_counter()
#     for img in batch.frames:
#         tensor_image_batch.append(
#             TF.to_tensor(img).cuda()
#         )
#     time_converting_to_tensor += time.perf_counter() - start_time
    
#     start_time = time.perf_counter()
#     sam_output = sam_harness(tensor_image_batch, "Person", move_to_cpu=False)
#     # sam_output = sam_harness(batch.frames, "Person", move_to_cpu=False)
#     time_in_sam += time.perf_counter() - start_time

#     start_time = time.perf_counter()
#     major_masks = []
#     for frame_out in sam_output:
#         major_mask_idx = torch.argmax(frame_out.masks.sum((1, 2)))
#         major_masks.append(frame_out.masks[major_mask_idx])
#     time_finding_best_mask += time.perf_counter() - start_time

#     start_time = time.perf_counter()
#     for idx in range(len(batch.frame_paths)):
#         frame_path = batch.frame_paths[idx]
#         parent_path = frame_path.parent
#         frame_stem = frame_path.stem
#         major_mask = major_masks[idx]

#         # Get the path in the sidecar dir
#         parent_rel_path = parent_path.relative_to(DUKEMTMC_VIDEO_REID_PATH)
#         sidecar_parent_path = DUKEMTMC_VIDEO_REID_SIDECAR_PATH / parent_rel_path
#         # Ensure our parent directory exists
#         sidecar_parent_path.mkdir(parents=True, exist_ok=True)

#         # Save the mask
#         sidecar_mask_path = sidecar_parent_path / f"{frame_stem}_major_mask.png"
#         # print(f"Saving mask to {sidecar_mask_path}")
#         float_mask = major_mask.float()
#         save_image(float_mask, sidecar_mask_path)
#     time_saving += time.perf_counter() - start_time
        

#     # print(batch.frame_paths)
#     # break